In [ ]:
{
  "mcpServers": {
    "docs-langchain": {
      "url": "https://docs.langchain.com/mcp"
    }
  }
}
from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="claude-sonnet-4-5-20250929",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

#Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)

In [5]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

@tool
def get_weather_for_location(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

@dataclass
class Context:
    """Custom runtime context schema."""
    user_id: str

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """Retrieve user information based on user ID."""
    user_id = runtime.context.user_id
    return "Florida" if user_id == "1" else "SF"

In [6]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "claude-sonnet-4-5-20250929",
    temperature=0.5,
    timeout=10,
    max_tokens=1000
)

In [7]:
from dataclasses import dataclass

# We use a dataclass here, but Pydantic models are also supported.
@dataclass
class ResponseFormat:
    """Response schema for the agent."""
    # A punny response (always required)
    punny_response: str
    # Any interesting information about the weather if available
    weather_conditions: str | None = None

In [8]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [1]:
from langchain.agents.structured_output import ToolStrategy

agent = create_agent(
    model=model,
    # system_prompt=SYSTEM_PROMPT,
    tools=[get_user_location, get_weather_for_location],
    context_schema=Context,
    response_format=ToolStrategy(ResponseFormat),
    checkpointer=checkpointer
)

# `thread_id` is a unique identifier for a given conversation.
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather outside?"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="Florida is still having a 'sun-derful' day! The sunshine is playing 'ray-dio' hits all day long! I'd say it's the perfect weather for some 'solar-bration'! If you were hoping for rain, I'm afraid that idea is all 'washed up' - the forecast remains 'clear-ly' brilliant!",
#     weather_conditions="It's always sunny in Florida!"
# )


# Note that we can continue the conversation using the same `thread_id`.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="You're 'thund-erfully' welcome! It's always a 'breeze' to help you stay 'current' with the weather. I'm just 'cloud'-ing around waiting to 'shower' you with more forecasts whenever you need them. Have a 'sun-sational' day in the Florida sunshine!",
#     weather_conditions=None
# )

NameError: name 'create_agent' is not defined

In [3]:
pip install -U langchain langchain-community


In [20]:
# from langchain_community.chat_models import ChatOllama
# from langchain.chains import ConversationChain
# from langchain.memory import ConversationBufferMemory

# llm = ChatOllama(
#     model="llama3",
#     temperature=0.7
# )

# memory = ConversationBufferMemory(
#     return_messages=True
# )

# chatbot = ConversationChain(
#     llm=llm,
#     memory=memory,
#     verbose=True
# )

# print("🤖 Chatbot started (type 'exit' to stop)\n")

# while True:
#     user_input = input("You: ")

#     if user_input.lower() == "exit":
#         print("Chatbot: Goodbye 👋")
#         break

#     response = chatbot.predict(input=user_input)
#     print("Chatbot:", response)


In [19]:
from langchain_community.chat_models import ChatOllama

llm = ChatOllama(model="llama3")

print(llm.invoke("Explain transformers using deep learning in simple words"))


content='Transformers! They\'re a type of AI model that\'s revolutionized the field of natural language processing (NLP). I\'ll try to explain them in simple terms, using analogies and examples.\n\n**What are transformers?**\n\nImagine you\'re trying to translate a sentence from English to Spanish. You know the words "Hello" means "Hola" in Spanish, but how do you get from the original sentence to the translated one?\n\nA transformer is like a super-smart translator that learns to "transform" (hence the name!) input sentences into output sentences by analyzing patterns and relationships between words.\n\n**How do transformers work?**\n\nHere\'s a simplified overview:\n\n1. **Input**: You give the transformer an input sentence, like "I love playing basketball."\n2. **Self-Attention Mechanism**: The transformer breaks down the input sentence into smaller chunks (called tokens) and looks at each token in relation to every other token. This is called self-attention. Think of it as taking a